# XGBoost Training — REFIT Dataset

Trains one XGBoost binary classifier per appliance (television, heating) on REFIT per-house Parquet files.

Mirrors the Plegma training script with one key difference: **time-based 80/20 split** (sorted by timestamp) instead of house-based, because REFIT houses tend to have fewer unique houses available.

## Configuration

In [1]:
import os
import glob
import pickle
import warnings
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score,
                             roc_auc_score, classification_report)

warnings.filterwarnings("ignore")

# ── 1. Config ─────────────────────────────────────────────────────────────────
REFIT_DIR   = r"C:\Users\moham\Documents\490 project new\refit_houses"
MODELS_DIR  = r"C:\Users\moham\Documents\490 project new\models_refit"
RANDOM_SEED = 42
MIN_HOUSES  = 3  # minimum house files needed to train a model

WANTED_APPLIANCES = [
    #'elec_television_on',
    'elec_heating_on',
]

FEATURE_COLS = [
    'weather_drybulb_temp_c',
    'weather_relative_humidity_pct',
    'hour',
    'day_of_week',
    'is_weekend',
    'month',
]

PARAM_GRID = {
    "n_estimators":  [100, 300],
    "max_depth":     [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.3],
}

## Load All House Files

In [2]:
os.makedirs(MODELS_DIR, exist_ok=True)

house_files = sorted(glob.glob(os.path.join(REFIT_DIR, '*.parquet')))
if not house_files:
    raise FileNotFoundError(f"No parquet files found in {REFIT_DIR}")

print(f"Found {len(house_files)} house files\n")

houses = {}
for f in house_files:
    df_h     = pd.read_parquet(f)
    house_id = df_h['house_id'].iloc[0]
    houses[house_id] = df_h
    print(f"  Loaded {house_id}: {len(df_h):,} rows | "
          f"appliances: {[c for c in df_h.columns if c in WANTED_APPLIANCES]}")

Found 25 house files

  Loaded House_01_heat: 15,335 rows | appliances: ['elec_heating_on']
  Loaded House_01_tv: 15,335 rows | appliances: []
  Loaded House_02_tv: 14,819 rows | appliances: []
  Loaded House_03_tv: 14,752 rows | appliances: []
  Loaded House_04_tv: 15,216 rows | appliances: []
  Loaded House_05_tv: 15,561 rows | appliances: []
  Loaded House_06_tv: 13,859 rows | appliances: []
  Loaded House_07_tv: 14,717 rows | appliances: []
  Loaded House_08_tv: 13,322 rows | appliances: []
  Loaded House_09_heat: 13,634 rows | appliances: ['elec_heating_on']
  Loaded House_09_tv: 13,634 rows | appliances: []
  Loaded House_10_tv: 14,089 rows | appliances: []
  Loaded House_12_tv1: 11,705 rows | appliances: []
  Loaded House_12_tv2: 11,705 rows | appliances: []
  Loaded House_13_tv1: 11,966 rows | appliances: []
  Loaded House_13_tv2: 11,966 rows | appliances: []
  Loaded House_15_tv: 13,618 rows | appliances: []
  Loaded House_16_heat1: 13,049 rows | appliances: ['elec_heating_on'

## Training Loop — One Model per Appliance

### Eligible house selection & combination

In [3]:
results = []

for target in WANTED_APPLIANCES:
    print(f"\n{'='*55}")
    print(f"APPLIANCE: {target}")
    print(f"{'='*55}")

    # Find house files that have this appliance column AND have ON events
    eligible_houses = {}
    for house_id, df_h in houses.items():
        if target not in df_h.columns:
            continue
        on_count = (df_h[target] == 1).sum()
        if on_count == 0:
            continue
        eligible_houses[house_id] = df_h

    n_eligible = len(eligible_houses)
    print(f"  Houses with this appliance: {n_eligible} "
          f"({list(eligible_houses.keys())})")

    if n_eligible < MIN_HOUSES:
        print(f"  [SKIPPED] Need at least {MIN_HOUSES} houses, only have {n_eligible}")
        continue

    # Combine only eligible houses
    df = pd.concat(
        [df_h[[*FEATURE_COLS, target, 'house_id', 'timestamp']]
         for df_h in eligible_houses.values()],
        ignore_index=True
    )

    # Drop any remaining NaN rows
    before = len(df)
    df     = df.dropna()
    if before != len(df):
        print(f"  Dropped {before - len(df):,} NaN rows")

    print(f"  Combined rows: {len(df):,}")

    # ── Time-based train/test split ───────────────────────────────────────────
    df = df.sort_values('timestamp').reset_index(drop=True)

    split      = int(len(df) * 0.8)
    train_mask = df.index < split
    test_mask  = df.index >= split

    X_train = df.loc[train_mask, FEATURE_COLS].reset_index(drop=True)
    X_test  = df.loc[test_mask,  FEATURE_COLS].reset_index(drop=True)
    y_train = df.loc[train_mask, target].astype(int).reset_index(drop=True)
    y_test  = df.loc[test_mask,  target].astype(int).reset_index(drop=True)
    groups  = df.loc[train_mask, 'house_id'].reset_index(drop=True)

    on_train = y_train.sum()
    on_test  = y_test.sum()
    print(f"  Train rows: {len(X_train):,} | ON: {on_train:,}")
    print(f"  Test  rows: {len(X_test):,}  | ON: {on_test:,}")

    if on_train == 0:
        print(f"  [SKIPPED] No ON events in training set")
        continue

    # ── Train ─────────────────────────────────────────────────────────────────
    neg = (y_train == 0).sum()
    pos = on_train
    scale_pos_weight = min(neg / pos, 5.0)
    print(f"  scale_pos_weight: {scale_pos_weight:.2f}")

    base_model = XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric="aucpr",
        random_state=RANDOM_SEED,
        device="cuda",
        n_jobs=-1,
    )

    n_splits    = min(5, df.loc[train_mask, 'house_id'].nunique())
    group_kfold = GroupKFold(n_splits=n_splits)
    cv_splits   = group_kfold.split(X_train, y_train, groups=groups)

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=PARAM_GRID,
        cv=cv_splits,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=0,
    )

    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_

    # ── Evaluate ──────────────────────────────────────────────────────────────
    y_pred  = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_test, y_proba)
    except ValueError:
        auc = float("nan")

    print(f"\n  Best params: {grid_search.best_params_}")
    print(f"  CV F1:       {grid_search.best_score_:.4f}")
    print(f"  Test Acc:    {acc:.4f} | Test F1: {f1:.4f} | AUC: {auc:.4f}")
    print(classification_report(y_test, y_pred, zero_division=0))

    results.append({
        "appliance":  target,
        "houses":     n_eligible,
        "cv_f1":      round(grid_search.best_score_, 4),
        "test_f1":    round(f1,  4),
        "test_auc":   round(auc, 4),
        "test_acc":   round(acc, 4),
        "best_params": grid_search.best_params_,

            })

    model_path = os.path.join(MODELS_DIR, f"{target}.pkl")
    with open(model_path, "wb") as f:
        pickle.dump(best_model, f)
    print(f"  Saved → {model_path}")


APPLIANCE: elec_heating_on
  Houses with this appliance: 5 (['House_01_heat', 'House_09_heat', 'House_16_heat1', 'House_16_heat2', 'House_16_heat3'])
  Combined rows: 68,116
  Train rows: 54,492 | ON: 2,807
  Test  rows: 13,624  | ON: 254
  scale_pos_weight: 5.00

  Best params: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 300}
  CV F1:       0.5020
  Test Acc:    0.9538 | Test F1: 0.1442 | AUC: 0.7526
              precision    recall  f1-score   support

           0       0.98      0.97      0.98     13370
           1       0.11      0.21      0.14       254

    accuracy                           0.95     13624
   macro avg       0.55      0.59      0.56     13624
weighted avg       0.97      0.95      0.96     13624

  Saved → C:\Users\moham\Documents\490 project new\models_refit\elec_heating_on.pkl


In [4]:
# ── 4. Summary ────────────────────────────────────────────────────────────────
if results:
    print(f"\n{'='*70}")
    print("SUMMARY")
    print(f"{'='*70}")
    summary = pd.DataFrame(results).drop(columns=["best_params"])
    print(summary.to_string(index=False))
else:
    print("\nNo models were trained.")


SUMMARY
      appliance  houses  cv_f1  test_f1  test_auc  test_acc
elec_heating_on       5  0.502   0.1442    0.7526    0.9538
